# PyTorch Mastery Lab

Цель: полностью освоить PyTorch-операции, необходимые для свободного чтения и написания
кода продвинутых моделей уровня DeepSeek V3.

Каждая секция следует циклу обучения:

| Шаг | Что | Иконка |
|-----|-----|--------|
| **Теория** | Объяснение концепции + формулы + диаграмма shapes | 📖 |
| **Пример** | Рабочий код, который вы запускаете и наблюдаете | 👀 |
| **Упражнение** | Скелет с `# YOUR CODE` | ✏️ |
| **Проверка** | Assert-ы, печатающие `OK` / `FAIL` | ✅ |
| **Советы** | «В DeepSeek этот паттерн используется в…» + подводные камни | 💡 |

**Соглашения по shapes:**

```text
B       = batch size
T / S   = sequence length
D       = model dimension (d_model)
H       = number of heads
D_head  = per-head dimension (D // H)
D_rope  = rotary embedding dimension
D_nope  = non-positional dimension
```

In [ ]:
from __future__ import annotations

import math

import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

print("PyTorch:", torch.__version__)


# ── Helpers ──────────────────────────────────────────────────
def preview(name: str, t: Tensor, rows: int = 4, cols: int = 8) -> None:
    print(f"{name}  shape={tuple(t.shape)}  dtype={t.dtype}")
    if t.ndim <= 2:
        print(t[:rows, :cols] if t.ndim == 2 else t[:cols])
    else:
        print(t)

def shape_eq(t: Tensor, expected: tuple) -> None:
    assert tuple(t.shape) == expected, f"Expected {expected}, got {tuple(t.shape)}"

def check(name: str, ok: bool) -> None:
    print(f"  {name}: {'OK ✅' if ok else 'FAIL ❌'}")

---
# Глава 1 — Tensor Foundations

> **Цель**: уверенное создание, инспекция, изменение формы тензоров,
> управление типами данных и устройствами.

## 1.1 Создание тензоров

📖 **Теория**

PyTorch предоставляет набор фабричных функций. Главные:

| Функция | Что создает |
|---------|-------------|
| `torch.empty(shape)` | Неинициализированная память (мусор) |
| `torch.zeros(shape)` | Нули |
| `torch.ones(shape)` | Единицы |
| `torch.full(shape, val)` | Константа `val` |
| `torch.randn(shape)` | Нормальное распределение N(0,1) |
| `torch.randint(lo, hi, shape)` | Целые числа в [lo, hi) |
| `torch.arange(start, end, step)` | Последовательность |
| `torch.*_like(other)` | Такой же shape/dtype/device |

Важный аргумент — `dtype`:
```python
torch.empty(3, 4, dtype=torch.bfloat16)
```

In [ ]:
# 👀 Пример: основные фабрики
z = torch.zeros(2, 3)
preview("zeros", z)

r = torch.randn(2, 3)
preview("randn", r)

seq = torch.arange(0, 10, 2)
preview("arange", seq)

like = torch.zeros_like(r)
preview("zeros_like", like)
print(f"  same shape: {like.shape == r.shape}, same dtype: {like.dtype == r.dtype}")

### ✏️ Упражнение 1.1a: Создайте тензоры по спецификации

1. `a` — матрица 4×8 из единиц, dtype=float32
2. `b` — вектор целых чисел от 0 до 99 (100 элементов)
3. `c` — тензор той же формы и dtype, что `a`, заполненный числом -1.0
4. `d` — случайные целые числа в [0, 102400), форма (2, 128) — имитация batch токенов

In [ ]:
# ✏️ YOUR CODE
a = ...  # (4, 8) float32 ones
b = ...  # (100,) integers 0..99
c = ...  # same shape as a, filled with -1.0
d = ...  # (2, 128) random ints in [0, 102400)

In [ ]:
# ✅ Проверка 1.1a
shape_eq(a, (4, 8)); check("a shape", True)
assert a.dtype == torch.float32; check("a dtype", True)
assert torch.all(a == 1.0); check("a values", True)

shape_eq(b, (100,)); check("b shape", True)
assert b[0] == 0 and b[99] == 99; check("b range", True)

shape_eq(c, (4, 8)); check("c shape", True)
assert torch.all(c == -1.0); check("c values", True)

shape_eq(d, (2, 128)); check("d shape", True)
assert d.dtype in (torch.int32, torch.int64); check("d dtype", True)
assert d.min() >= 0 and d.max() < 102400; check("d range", True)
print("\n1.1a: all OK ✅")

### ✏️ Упражнение 1.1b: Паттерн KV-кэша

В DeepSeek `MLA.__init__` выделяется кэш для ключей и значений:
```python
self.register_buffer("kv_cache",
    torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank),
    persistent=False)
```

Создайте аналогичный тензор-кэш:

In [ ]:
# ✏️ YOUR CODE
max_batch_size = 8
max_seq_len = 4096
kv_lora_rank = 512

kv_cache = ...  # shape (8, 4096, 512), zeros

In [ ]:
# ✅ Проверка 1.1b
shape_eq(kv_cache, (8, 4096, 512)); check("kv_cache shape", True)
assert torch.all(kv_cache == 0); check("kv_cache zeros", True)
print(f"Memory: {kv_cache.element_size() * kv_cache.numel() / 1e6:.1f} MB")
print("\n1.1b: OK ✅")

💡 **Совет**: `torch.empty` быстрее `torch.zeros`, но содержит мусор.
В DeepSeek `Linear.__init__` использует `torch.empty` для весов
(т.к. они будут загружены из чекпоинта), а `torch.zeros` — для кэшей
(т.к. кэш должен стартовать с нулей).

## 1.2 Shape, Size, Numel, Ndim

📖 **Теория**

| Метод | Что возвращает | Пример для (2,4,3) |
|-------|---------------|-------------------|
| `.shape` | `torch.Size` (tuple-like) | `torch.Size([2, 4, 3])` |
| `.size()` | то же самое | `torch.Size([2, 4, 3])` |
| `.size(dim)` | int по оси | `.size(1)` → `4` |
| `.numel()` | общее число элементов | `24` |
| `.ndim` | число измерений | `3` |

В DeepSeek `.size()` используется повсеместно:
```python
bsz, seqlen, _ = x.size()     # распаковка трёх измерений
x.size(0)                      # batch size
x.size(1)                      # sequence length
x.size(-1)                     # last dimension
```

In [ ]:
# 👀 Пример
x = torch.randn(2, 4, 3)
print(f"x.shape      = {x.shape}")
print(f"x.size()     = {x.size()}")
print(f"x.size(1)    = {x.size(1)}")
print(f"x.numel()    = {x.numel()}")
print(f"x.ndim       = {x.ndim}")

# Распаковка — паттерн из DeepSeek MLA.forward:
bsz, seqlen, dim = x.size()
print(f"\nbsz={bsz}, seqlen={seqlen}, dim={dim}")

### ✏️ Упражнение 1.2: Предскажите и извлеките

Дан тензор `t` формы `(B, T, H, D_head)`. Извлеките каждую размерность.

In [ ]:
# ✏️ YOUR CODE
B, T, H, D_head = 2, 8, 16, 64
t = torch.randn(B, T, H, D_head)

# Извлеките значения из t (не из переменных выше!)
batch_size = ...
seq_len = ...
n_heads = ...
head_dim = ...
total_elements = ...

In [ ]:
# ✅ Проверка 1.2
assert batch_size == 2; check("batch_size", True)
assert seq_len == 8; check("seq_len", True)
assert n_heads == 16; check("n_heads", True)
assert head_dim == 64; check("head_dim", True)
assert total_elements == 2 * 8 * 16 * 64; check("total_elements", True)
print("\n1.2: OK ✅")

💡 **Совет**: `.size(dim)` возвращает `int`, а `.shape[dim]` — тоже `int`.
Они полностью взаимозаменяемы. DeepSeek использует `.size()` для распаковки
и `.size(dim)` для отдельных осей — это самый распространённый стиль.

## 1.3 View, Reshape, Flatten

📖 **Теория**

`view` переинтерпретирует память без копирования. Правило:
**число элементов должно совпадать**.

```text
x.view(new_shape)        # требует contiguous
x.reshape(new_shape)     # работает всегда (может копировать)
x.flatten(start_dim)     # сплющивает с start_dim до конца
```

Ключевые паттерны из DeepSeek:
```python
# Head split: (B, T, H*D_head) → (B, T, H, D_head)
q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)

# Head merge: (B, T, H, D_head) → (B, T, H*D_head)
x = x.flatten(2)

# MoE flatten: (B, T, D) → (B*T, D)
x = x.view(-1, self.dim)
```

`-1` означает «вычисли автоматически».

In [ ]:
# 👀 Пример: head split и merge
B, T, H, D_head = 2, 8, 16, 64
D = H * D_head  # 1024

q_flat = torch.randn(B, T, D)  # (2, 8, 1024)
print(f"Before split: {q_flat.shape}")

q_heads = q_flat.view(B, T, H, D_head)  # (2, 8, 16, 64)
print(f"After split:  {q_heads.shape}")

q_merged = q_heads.flatten(2)  # (2, 8, 1024)
print(f"After merge:  {q_merged.shape}")

assert torch.allclose(q_flat, q_merged)
print("Round-trip: identical ✅")

### ✏️ Упражнение 1.3a: Head split

Реализуйте функцию, которая разделяет `(B, T, H*D_head)` → `(B, T, H, D_head)`.

In [ ]:
# ✏️ YOUR CODE
def head_split(x: Tensor, n_heads: int) -> Tensor:
    """(B, T, H*D_head) -> (B, T, H, D_head)"""
    raise NotImplementedError

In [ ]:
# ✅ Проверка 1.3a
B, T, H, D_head = 2, 8, 16, 64
q = torch.randn(B, T, H * D_head)
q_split = head_split(q, H)
shape_eq(q_split, (B, T, H, D_head)); check("head_split shape", True)
assert torch.allclose(q, q_split.flatten(2)); check("round-trip", True)
print("\n1.3a: OK ✅")

### ✏️ Упражнение 1.3b: MoE flatten и restore

В MoE блоке DeepSeek:
```python
shape = x.size()          # save original shape
x = x.view(-1, self.dim)  # flatten to 2D for expert routing
...
return (y + z).view(shape) # restore
```

Реализуйте обе операции.

In [ ]:
# ✏️ YOUR CODE
def moe_flatten(x: Tensor) -> tuple[Tensor, torch.Size]:
    """(B, T, D) -> ((B*T, D), original_shape)"""
    raise NotImplementedError

def moe_restore(x: Tensor, original_shape: torch.Size) -> Tensor:
    """(B*T, D) -> (B, T, D) using saved shape"""
    raise NotImplementedError

In [ ]:
# ✅ Проверка 1.3b
B, T, D = 4, 32, 2048
x = torch.randn(B, T, D)
x_flat, orig = moe_flatten(x)
shape_eq(x_flat, (B * T, D)); check("flatten shape", True)
x_back = moe_restore(x_flat, orig)
shape_eq(x_back, (B, T, D)); check("restore shape", True)
assert torch.allclose(x, x_back); check("round-trip", True)
print("\n1.3b: OK ✅")

### ✏️ Упражнение 1.3c: Полный MLA view-паттерн

В MLA Q проходит путь:
1. Linear projection: `(B, T, D)` → `(B, T, H * qk_head_dim)`
2. View to heads: → `(B, T, H, qk_head_dim)`
3. Split nope/rope: → `(B, T, H, qk_nope)` + `(B, T, H, qk_rope)`

Реализуйте шаги 2 и 3.

In [ ]:
# ✏️ YOUR CODE
B, T, H = 2, 8, 16
qk_nope_dim = 128
qk_rope_dim = 64
qk_head_dim = qk_nope_dim + qk_rope_dim  # 192

# Simulated output of wq projection
q_proj = torch.randn(B, T, H * qk_head_dim)

# Step 2: reshape to heads
q_heads = ...  # (B, T, H, 192)

# Step 3: split into nope and rope parts
q_nope = ...   # (B, T, H, 128)
q_pe = ...     # (B, T, H, 64)

In [ ]:
# ✅ Проверка 1.3c
shape_eq(q_heads, (B, T, H, qk_head_dim)); check("q_heads shape", True)
shape_eq(q_nope, (B, T, H, qk_nope_dim)); check("q_nope shape", True)
shape_eq(q_pe, (B, T, H, qk_rope_dim)); check("q_pe shape", True)
recombined = torch.cat([q_nope, q_pe], dim=-1).flatten(2)
assert torch.allclose(q_proj, recombined); check("round-trip", True)
print("\n1.3c: OK ✅")

💡 **Совет**: `view` — zero-copy, `reshape` — может копировать.
В production-коде DeepSeek используется `view`, а `flatten(start_dim)` —
это просто сахар для `view(..., -1)` начиная с `start_dim`.
Когда пишете свой Transformer, привыкайте к `view` — он вам явно покажет ошибку,
если тензор стал non-contiguous после transpose.

## 1.4 Squeeze, Unsqueeze и `None`-indexing

📖 **Теория**

| Операция | Что делает | Пример |
|----------|-----------|--------|
| `unsqueeze(dim)` | Вставляет ось длины 1 в позицию `dim` | `(T,D)` → `.unsqueeze(0)` → `(1,T,D)` |
| `squeeze(dim)` | Удаляет ось длины 1 в позиции `dim` | `(1,T,1,D)` → `.squeeze(0)` → `(T,1,D)` |
| `squeeze()` | Удаляет **все** оси длины 1 | `(1,T,1,D)` → `(T,D)` |
| `x[None]` | Эквивалент `unsqueeze(0)` | `(T,D)` → `(1,T,D)` |
| `x[:, None]` | Эквивалент `unsqueeze(1)` | `(T,D)` → `(T,1,D)` |
| `x[..., None]` | Эквивалент `unsqueeze(-1)` | `(T,D)` → `(T,D,1)` |

Зачем? **Broadcasting.** Чтобы сложить `(B, T, D) + (T, D)`, нужно привести
`(T, D)` к `(1, T, D)`. PyTorch делает это автоматически в простых случаях,
но для einsum, matmul и expand приходится делать руками.

Паттерны из DeepSeek:
```python
k_pe.unsqueeze(2)                              # (B,T,D) → (B,T,1,D) для rotary
freqs_cis.view(1, x.size(1), 1, x.size(-1))    # вставить batch и head dims
mask.unsqueeze(1)                               # (B,T,T) → (B,1,T,T) для broadcast по heads
weights[idx, top, None]                         # скалярный вес → (N,1) для broadcast
```

In [ ]:
# 👀 Пример: unsqueeze и None-indexing
x = torch.randn(4, 8)  # (T, D)
print(f"Original:            {x.shape}")
print(f"unsqueeze(0):        {x.unsqueeze(0).shape}")
print(f"unsqueeze(1):        {x.unsqueeze(1).shape}")
print(f"unsqueeze(-1):       {x.unsqueeze(-1).shape}")
print(f"x[None]:             {x[None].shape}")
print(f"x[:, None]:          {x[:, None].shape}")
print(f"x[..., None]:        {x[..., None].shape}")

# Verify equivalence
assert torch.equal(x.unsqueeze(0), x[None])
assert torch.equal(x.unsqueeze(1), x[:, None])
assert torch.equal(x.unsqueeze(-1), x[..., None])
print("\nAll equivalent ✅")

### ✏️ Упражнение 1.4a: unsqueeze фундаментальные

Дан тензор `v` формы `(T, D)`. Создайте три версии с добавленной осью.

In [ ]:
# ✏️ YOUR CODE
T, D = 8, 64
v = torch.randn(T, D)

v_batch = ...  # (1, T, D) — добавить batch dimension
v_head = ...   # (T, 1, D) — добавить head dimension
v_last = ...   # (T, D, 1) — добавить trailing dimension

In [ ]:
# ✅ Проверка 1.4a
shape_eq(v_batch, (1, T, D)); check("v_batch shape", True)
shape_eq(v_head, (T, 1, D)); check("v_head shape", True)
shape_eq(v_last, (T, D, 1)); check("v_last shape", True)
print("\n1.4a: OK ✅")

### ✏️ Упражнение 1.4b: `None`-indexing эквивалентность

Тот же результат, но через `None`-indexing.

In [ ]:
# ✏️ YOUR CODE
v_batch_none = ...  # (1, T, D)
v_head_none = ...   # (T, 1, D)
v_last_none = ...   # (T, D, 1)

In [ ]:
# ✅ Проверка 1.4b
assert torch.equal(v_batch, v_batch_none); check("batch equiv", True)
assert torch.equal(v_head, v_head_none); check("head equiv", True)
assert torch.equal(v_last, v_last_none); check("last equiv", True)
print("\n1.4b: OK ✅")

### ✏️ Упражнение 1.4c: squeeze назад

Дан тензор `padded` формы `(1, T, 1, D)`.

1. Сожмите его до `(T, D)` одним вызовом.
2. Сожмите только `dim=0`, получив `(T, 1, D)`.

Объясните разницу между `squeeze()` и `squeeze(dim)`.

In [ ]:
# ✏️ YOUR CODE
T, D = 8, 64
padded = torch.randn(1, T, 1, D)

squeezed_all = ...   # (T, D)
squeezed_dim0 = ...  # (T, 1, D)

In [ ]:
# ✅ Проверка 1.4c
shape_eq(squeezed_all, (T, D)); check("squeeze all", True)
shape_eq(squeezed_dim0, (T, 1, D)); check("squeeze dim0", True)
print("\n1.4c: OK ✅")

### ✏️ Упражнение 1.4d: Broadcasting для freqs_cis

В `apply_rotary_emb` DeepSeek:
```python
freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))
```

Это превращает `(T, D_rope)` в `(1, T, 1, D_rope)`, чтобы broadcast
с Q/K формы `(B, T, H, D_rope)`.

Реализуйте эту трансформацию через `unsqueeze`.

In [ ]:
# ✏️ YOUR CODE
T, D_rope = 128, 64
B, H = 2, 16

freqs = torch.randn(T, D_rope)

# Вариант 1: через view (как в DeepSeek)
freqs_v1 = ...  # (1, T, 1, D_rope)

# Вариант 2: через unsqueeze (два вызова)
freqs_v2 = ...  # (1, T, 1, D_rope)

# Проверка: broadcast с Q
q_rope = torch.randn(B, T, H, D_rope)
result = q_rope * freqs_v1  # должно broadcast

In [ ]:
# ✅ Проверка 1.4d
shape_eq(freqs_v1, (1, T, 1, D_rope)); check("freqs_v1 shape", True)
shape_eq(freqs_v2, (1, T, 1, D_rope)); check("freqs_v2 shape", True)
assert torch.equal(freqs_v1, freqs_v2); check("v1 == v2", True)
shape_eq(result, (B, T, H, D_rope)); check("broadcast result shape", True)
print("\n1.4d: OK ✅")

### ✏️ Упражнение 1.4e: Маска внимания broadcasting

В DeepSeek `Transformer.forward`:
```python
mask = torch.full((seqlen, seqlen), float("-inf")).triu_(1)
```
Потом в MLA:
```python
scores += mask.unsqueeze(1)  # (T,T) broadcast with (B,S,H,T)
```

Но если scores имеют форму `(B, H, T, T)` (стандартная attention),
нужно `(1, 1, T, T)`.

Создайте causal mask и добавьте нужные оси.

In [ ]:
# ✏️ YOUR CODE
T = 8
B, H = 2, 4

# 1. Создайте causal mask (T, T) с -inf выше диагонали
causal_mask = ...

# 2. Расширьте до (1, 1, T, T) для broadcast с (B, H, T, T)
mask_4d = ...

# 3. Создайте dummy scores и примените маску
scores = torch.randn(B, H, T, T)
masked_scores = scores + mask_4d

In [ ]:
# ✅ Проверка 1.4e
shape_eq(causal_mask, (T, T)); check("causal_mask shape", True)
shape_eq(mask_4d, (1, 1, T, T)); check("mask_4d shape", True)
shape_eq(masked_scores, (B, H, T, T)); check("masked_scores shape", True)

# Верхний треугольник (без диагонали) должен быть -inf
assert causal_mask[0, 1] == float("-inf"); check("upper triangle is -inf", True)
assert causal_mask[1, 0] == 0.0; check("lower triangle is 0", True)
assert causal_mask[0, 0] == 0.0; check("diagonal is 0", True)
print("\n1.4e: OK ✅")

### ✏️ Упражнение 1.4f: Трюк `[..., None]` для скалярных весов

В MoE dispatch DeepSeek:
```python
y[idx] += expert(x[idx]) * weights[idx, top, None]
```

`weights[idx, top]` — скаляр (вес эксперта), а `expert(x[idx])` — вектор `(D,)`.
`None` добавляет ось, превращая скаляр в `(1,)` для broadcasting.

Реализуйте этот паттерн.

In [ ]:
# ✏️ YOUR CODE
N, D = 5, 128  # 5 tokens routed to this expert

expert_output = torch.randn(N, D)        # (5, 128)
routing_weights = torch.rand(N)           # (5,) — per-token weights

# Масштабируйте каждую строку expert_output соответствующим весом
# НЕ используйте цикл!
weighted_output = ...  # (5, 128)

In [ ]:
# ✅ Проверка 1.4f
shape_eq(weighted_output, (N, D)); check("shape", True)

# Проверка: первая строка = expert_output[0] * routing_weights[0]
expected_row0 = expert_output[0] * routing_weights[0]
assert torch.allclose(weighted_output[0], expected_row0); check("row 0 scaling", True)

# Проверка: последняя строка
expected_last = expert_output[-1] * routing_weights[-1]
assert torch.allclose(weighted_output[-1], expected_last); check("last row scaling", True)
print("\n1.4f: OK ✅")

💡 **Совет**: `None`-indexing — самый компактный способ unsqueeze.
В DeepSeek он используется именно в одном месте — MoE dispatch — но
это критически важный паттерн. Запомните: если нужно помножить вектор
весов `(N,)` на матрицу `(N, D)`, используйте `weights[:, None]` или
`weights.unsqueeze(-1)` — оба дают `(N, 1)`, который broadcast до `(N, D)`.

## 1.5 Типы данных и приведение типов

📖 **Теория**

| Тип | Размер | Описание |
|-----|--------|----------|
| `torch.float32` | 4 bytes | Стандартная точность |
| `torch.bfloat16` | 2 bytes | Brain float — такая же экспонента как f32, меньше мантиссы |
| `torch.float16` | 2 bytes | Half precision |
| `torch.float8_e4m3fn` | 1 byte | FP8 для inference (DeepSeek) |
| `torch.int64` / `torch.long` | 8 bytes | Индексы |

Методы приведения:
```python
x.float()          # → float32
x.half()           # → float16
x.bfloat16()       # → bfloat16
x.to(dtype=...)    # универсальный
x.type_as(other)   # взять dtype и device от other
```

Паттерн из DeepSeek — **upcast для точности, потом downcast**:
```python
scores = scores.softmax(dim=-1, dtype=torch.float32).type_as(x)
```
Softmax считается в float32 (для численной стабильности), результат возвращается в bf16.

In [ ]:
# 👀 Пример: типы и память
x_f32 = torch.randn(1000, 1000)
x_bf16 = x_f32.bfloat16()

print(f"float32: {x_f32.element_size()} bytes/element, total {x_f32.element_size() * x_f32.numel() / 1e6:.1f} MB")
print(f"bfloat16: {x_bf16.element_size()} bytes/element, total {x_bf16.element_size() * x_bf16.numel() / 1e6:.1f} MB")
print(f"Max abs diff: {(x_f32 - x_bf16.float()).abs().max():.6f}")

# type_as pattern
y = torch.randn(3, dtype=torch.bfloat16)
z = torch.tensor([1.0, 2.0, 3.0])  # float32
z_matched = z.type_as(y)
print(f"\nz dtype: {z.dtype} → z_matched dtype: {z_matched.dtype}")

### ✏️ Упражнение 1.5: Softmax upcast/downcast

Реализуйте паттерн upcast → softmax → downcast из DeepSeek.

In [ ]:
# ✏️ YOUR CODE
scores = torch.randn(2, 4, 8, 8, dtype=torch.bfloat16)  # (B, H, T, T) in bf16

# 1. Примените softmax по dim=-1, вычисляя в float32
# 2. Верните результат обратно в bfloat16
attention_weights = ...

# Бонус: покажите, почему upcast важен
# Посчитайте softmax прямо в bf16 для сравнения
attention_weights_naive = ...

In [ ]:
# ✅ Проверка 1.5
shape_eq(attention_weights, (2, 4, 8, 8)); check("shape", True)
assert attention_weights.dtype == torch.bfloat16; check("dtype bf16", True)
row_sums = attention_weights.float().sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-3); check("rows sum to 1", True)
print("\n1.5: OK ✅")

💡 **Совет**: `element_size()` возвращает число байтов на элемент.
Это удобно для расчёта памяти. В DeepSeek `element_size()` используется
для определения, квантизован ли тензор:
```python
if weight.element_size() > 1:  # не FP8
    return F.linear(x, weight, bias)
```

## 1.6 Управление устройствами (CPU ↔ GPU ↔ NumPy)

📖 **Теория**

```python
x.to("cuda")           # → GPU
x.to("cpu")            # → CPU
x.cpu()                # → CPU (shortcut)
x.cuda()               # → GPU (shortcut)
x.numpy()              # → NumPy array (CPU only!)
torch.from_numpy(arr)  # NumPy → Tensor

# Глобальные настройки (из DeepSeek __main__):
torch.set_default_dtype(torch.bfloat16)
torch.set_default_device("cuda")

# Создать на том же устройстве:
y = torch.zeros(3, device=x.device)
```

### ✏️ Упражнение 1.6: Round-trip CPU → NumPy → Tensor

1. Создайте тензор, конвертируйте в NumPy, обратно в тензор.
2. Убедитесь, что данные идентичны.

In [ ]:
# ✏️ YOUR CODE
import numpy as np

original = torch.tensor([1.0, 2.0, 3.0, 4.0])

# 1. Tensor → NumPy
as_numpy = ...

# 2. NumPy → Tensor
back_to_tensor = ...

# 3. Создайте тензор на том же устройстве, что и original
same_device = ...  # zeros of shape (4,) on same device as original

In [ ]:
# ✅ Проверка 1.6
assert isinstance(as_numpy, np.ndarray); check("is numpy", True)
assert torch.is_tensor(back_to_tensor); check("is tensor", True)
assert torch.allclose(original, back_to_tensor); check("round-trip", True)
assert same_device.device == original.device; check("same device", True)
print("\n1.6: OK ✅")

💡 **Совет**: `x.numpy()` делит память с тензором (zero-copy на CPU).
Изменение numpy-массива изменит тензор! Используйте `.numpy().copy()`
если нужна независимая копия. На GPU используйте `x.cpu().numpy()`.

---
# Глава 2 — Operations & Linear Algebra

> **Цель**: свободно владеть element-wise операциями, reductions, matmul, einsum —
> всем, что делает attention и MoE рабочими.

## 2.1 Поэлементные операции

📖 **Теория**

Все арифметические операторы в PyTorch работают поэлементно:
`+`, `-`, `*`, `/`, `**`, а также функции `torch.exp`, `torch.log`, `torch.abs`.

Ключевой паттерн — **gated activation** (SwiGLU) из DeepSeek MLP:
```python
return self.w2(F.silu(self.w1(x)) * self.w3(x))
```

Разберём по шагам:
1. `self.w1(x)` → gate signal, shape `(B, T, inter_dim)`
2. `F.silu(...)` → применить SiLU activation
3. `self.w3(x)` → value signal, shape `(B, T, inter_dim)`
4. `* ` → поэлементное произведение (gating)
5. `self.w2(...)` → проекция обратно в `(B, T, dim)`

SiLU (Sigmoid Linear Unit): `silu(x) = x * sigmoid(x)`

In [ ]:
# 👀 Пример: SiLU и gated activation
x = torch.linspace(-3, 3, 7)

# SiLU = x * sigmoid(x)
silu_manual = x * torch.sigmoid(x)
silu_pytorch = F.silu(x)

print(f"x:            {x.tolist()}")
print(f"SiLU manual:  {silu_manual.tolist()}")
print(f"SiLU pytorch: {silu_pytorch.tolist()}")
assert torch.allclose(silu_manual, silu_pytorch, atol=1e-6)
print("Equivalent ✅")

### ✏️ Упражнение 2.1: Gated MLP (SwiGLU)

Реализуйте SwiGLU forward pass вручную (без nn.Linear — просто матрицы).

In [ ]:
# ✏️ YOUR CODE
B, T, D, inter_dim = 2, 4, 64, 128

x = torch.randn(B, T, D)

# Веса (используем random для демо)
w1 = torch.randn(inter_dim, D)  # gate projection
w3 = torch.randn(inter_dim, D)  # value projection
w2 = torch.randn(D, inter_dim)  # output projection

# Реализуйте SwiGLU: w2(silu(w1(x)) * w3(x))
# Подсказка: F.linear(x, weight) = x @ weight.T
gate = ...       # (B, T, inter_dim) — F.linear(x, w1)
gate_act = ...   # (B, T, inter_dim) — F.silu(gate)
value = ...      # (B, T, inter_dim) — F.linear(x, w3)
gated = ...      # (B, T, inter_dim) — element-wise product
output = ...     # (B, T, D) — F.linear(gated, w2)

In [ ]:
# ✅ Проверка 2.1
shape_eq(gate, (B, T, inter_dim)); check("gate shape", True)
shape_eq(gated, (B, T, inter_dim)); check("gated shape", True)
shape_eq(output, (B, T, D)); check("output shape", True)

# Verify against direct computation
expected = F.linear(F.silu(F.linear(x, w1)) * F.linear(x, w3), w2)
assert torch.allclose(output, expected, atol=1e-5); check("matches F.linear chain", True)
print("\n2.1: OK ✅")

💡 **Совет**: SwiGLU стала стандартной активацией в современных LLM
(LLaMA, DeepSeek, Gemma). Она требует **два** linear слоя параллельно (w1 и w3),
поэтому `inter_dim` обычно ~2/3 от «наивного» 4*dim, чтобы сохранить общий
параметрический бюджет.

## 2.2 Broadcasting: глубокое погружение

📖 **Теория**

Broadcasting Rules (от конца):
1. Если у тензора меньше измерений — дополнить единицами **слева**
2. Размерности должны совпадать или одна из них == 1
3. Размерность 1 «растягивается» до другой

```text
(B, T, H, D) * (1, T, 1, D) → (B, T, H, D)  ✅
(B, T, H, D) * (T, D)       → ошибка!        ❌  (T≠H, D≠D по выравниванию)
(B, T, D)    + (D,)          → (B, T, D)      ✅  (добавится (1,1,D))
```

`expand` — явный broadcast без копирования:
```python
k_pe.expand(-1, -1, self.n_local_heads, -1)
# (B, T, 1, D) → (B, T, H, D) — -1 = "оставить как есть"
```

In [ ]:
# 👀 Пример: broadcasting визуализация
a = torch.tensor([[1], [2], [3]])     # (3, 1)
b = torch.tensor([10, 20, 30, 40])   # (4,)
c = a + b                             # (3, 4)
print(f"a shape: {a.shape}")
print(f"b shape: {b.shape}")
print(f"a + b shape: {c.shape}")
print(c)

### ✏️ Упражнение 2.2: expand для multi-head keys

В DeepSeek MLA key_pe имеет одну голову, но нужно расширить до H голов:
```python
k_pe.expand(-1, -1, self.n_local_heads, -1)
```

Реализуйте этот паттерн.

In [ ]:
# ✏️ YOUR CODE
B, T, H, D_rope = 2, 8, 16, 64

# k_pe изначально (B, T, 1, D_rope) — одна "голова"
k_pe = torch.randn(B, T, 1, D_rope)

# Расширьте до (B, T, H, D_rope) используя expand
k_pe_expanded = ...

# Проверьте, что expand НЕ копирует данные (shares memory)
shares_memory = ...

In [ ]:
# ✅ Проверка 2.2
shape_eq(k_pe_expanded, (B, T, H, D_rope)); check("expanded shape", True)

# Все головы должны быть идентичны (expand просто повторяет вид)
assert torch.equal(k_pe_expanded[:, :, 0, :], k_pe_expanded[:, :, 5, :]); check("heads identical", True)
assert torch.equal(k_pe_expanded[:, :, 0, :], k_pe.squeeze(2)); check("matches original", True)

# expand shares memory (data_ptr одинаковый)
assert k_pe_expanded.data_ptr() == k_pe.data_ptr(); check("shares memory", True)
print("\n2.2: OK ✅")

💡 **Совет**: `expand` никогда не копирует данные, но `repeat` — копирует.
В DeepSeek expand используется для ключей (один набор на все головы),
что экономит ~16x памяти в кэше. Это суть MLA (Multi-Head Latent Attention) —
сжатие KV через low-rank, а не хранение полных K,V для каждой головы.

## 2.3 Reductions (свёртки по осям)

📖 **Теория**

| Метод | Что делает | `keepdim` |
|-------|-----------|-----------|
| `.sum(dim)` | Сумма по оси | Сохраняет ось длины 1 |
| `.mean(dim)` | Среднее | |
| `.max(dim)` | Максимум + argmax | Возвращает namedtuple |
| `.amax(dim)` | Только максимум | |
| `.norm(p, dim)` | Lp-норма | |

Важно: без `keepdim=True` ось исчезает:
```python
x = torch.randn(3, 4)
x.sum(dim=1).shape         # (3,)
x.sum(dim=1, keepdim=True).shape  # (3, 1) — удобно для broadcast обратно
```

Паттерны из DeepSeek:
```python
scores.amax(dim=-1)                    # Gate: макс score в группе
weights.sum(dim=-1, keepdim=True)      # нормализация весов
```

In [ ]:
# 👀 Пример
x = torch.tensor([[1., 2., 3.],
                   [4., 5., 6.]])

print(f"sum(dim=1):          {x.sum(dim=1)}")
print(f"sum(dim=1, keepdim): {x.sum(dim=1, keepdim=True).shape}")
print(f"amax(dim=-1):        {x.amax(dim=-1)}")
print(f"norm(2, dim=1):      {x.norm(2, dim=1)}")

### ✏️ Упражнение 2.3: Gate нормализация весов

В DeepSeek Gate (sigmoid scoring):
```python
weights /= weights.sum(dim=-1, keepdim=True)
```

Реализуйте нормализацию, чтобы веса по каждому токену суммировались в 1.

In [ ]:
# ✏️ YOUR CODE
N = 10          # tokens
topk = 6        # activated experts

# Случайные веса для topk экспертов на каждый токен
weights = torch.rand(N, topk)  # (10, 6)

# Нормализуйте так, чтобы каждая строка суммировалась в 1
weights_normalized = ...

In [ ]:
# ✅ Проверка 2.3
shape_eq(weights_normalized, (N, topk)); check("shape", True)
row_sums = weights_normalized.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones(N), atol=1e-6); check("rows sum to 1", True)
assert (weights_normalized >= 0).all(); check("all non-negative", True)
print("\n2.3: OK ✅")

💡 **Совет**: `keepdim=True` — ваш лучший друг при нормализации.
Без него вы получите shape mismatch при делении. Альтернатива —
явный unsqueeze после reduction, но `keepdim` чище.

## 2.4 Функции активации

📖 **Теория**

| Функция | Формула | Использование |
|---------|---------|---------------|
| `sigmoid(x)` | 1/(1+e^(-x)) | Гейтинг, бинарная вероятность |
| `softmax(x, dim)` | e^(x_i) / Σe^(x_j) | Attention weights, routing scores |
| `silu(x)` | x·sigmoid(x) | Gated MLP в LLM |
| `relu(x)` | max(0, x) | Классические сети |

Softmax — **самая важная** для Transformer:
```python
# Численно стабильная версия:
# softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))
```

### ✏️ Упражнение 2.4a: Softmax вручную

Реализуйте численно стабильный softmax по `dim=-1`.

In [ ]:
# ✏️ YOUR CODE
def my_softmax(x: Tensor, dim: int = -1) -> Tensor:
    """Numerically stable softmax."""
    raise NotImplementedError

In [ ]:
# ✅ Проверка 2.4a
scores = torch.randn(2, 4, 8, 8)
mine = my_softmax(scores, dim=-1)
ref = F.softmax(scores, dim=-1)
assert torch.allclose(mine, ref, atol=1e-6); check("matches F.softmax", True)
assert torch.allclose(mine.sum(dim=-1), torch.ones_like(mine.sum(dim=-1))); check("rows sum to 1", True)

# Тест с экстремальными значениями (стабильность)
extreme = torch.tensor([[1000., 1001., 1002.]])
mine_ext = my_softmax(extreme, dim=-1)
ref_ext = F.softmax(extreme, dim=-1)
assert torch.allclose(mine_ext, ref_ext, atol=1e-6); check("stable with large values", True)
print("\n2.4a: OK ✅")

### ✏️ Упражнение 2.4b: SiLU вручную

Реализуйте SiLU: `silu(x) = x * sigmoid(x)`

In [ ]:
# ✏️ YOUR CODE
def my_silu(x: Tensor) -> Tensor:
    raise NotImplementedError

In [ ]:
# ✅ Проверка 2.4b
x = torch.randn(3, 4)
assert torch.allclose(my_silu(x), F.silu(x), atol=1e-6); check("matches F.silu", True)
print("\n2.4b: OK ✅")

💡 **Совет**: В DeepSeek softmax вычисляется в float32 для стабильности:
`scores.softmax(dim=-1, dtype=torch.float32)`. Аргумент `dtype` в PyTorch
softmax делает upcast автоматически — это удобнее, чем писать
`.float().softmax(...).type_as(...)`.

## 2.5 Matrix Multiplication (`@`, `F.linear`, `torch.matmul`)

📖 **Теория**

| Операция | Shapes | Результат |
|----------|--------|-----------|
| `A @ B` | (M,K) @ (K,N) | (M,N) |
| `A @ B` (batched) | (B,M,K) @ (B,K,N) | (B,M,N) |
| `F.linear(x, W, b)` | x:(∗,K), W:(N,K) | x @ W.T + b = (∗,N) |

Почему `F.linear` важен: вес хранится как `(out, in)`, а не `(in, out)`.
```python
# Эти три записи эквивалентны:
F.linear(x, W, b)
x @ W.T + b
(W @ x.T).T + b
```

В DeepSeek:
```python
def linear(x, weight, bias=None):
    return F.linear(x, weight, bias)
```

In [ ]:
# 👀 Пример: F.linear == x @ W.T + b
D_in, D_out = 64, 128
x = torch.randn(2, 4, D_in)  # (B, T, D_in)
W = torch.randn(D_out, D_in) # (D_out, D_in)
b = torch.randn(D_out)

y1 = F.linear(x, W, b)
y2 = x @ W.T + b

print(f"F.linear shape: {y1.shape}")
print(f"x @ W.T + b shape: {y2.shape}")
assert torch.allclose(y1, y2, atol=1e-5)
print("Equivalent ✅")

### ✏️ Упражнение 2.5a: Докажите эквивалентность F.linear

Вычислите `F.linear(x, W, b)` тремя разными способами и покажите, что результаты совпадают.

In [ ]:
# ✏️ YOUR CODE
D_in, D_out = 32, 64
x = torch.randn(2, 8, D_in)
W = torch.randn(D_out, D_in)
b = torch.randn(D_out)

way1 = F.linear(x, W, b)              # PyTorch builtin
way2 = ...                             # x @ W.T + b
way3 = ...                             # torch.matmul(x, W.T) + b

In [ ]:
# ✅ Проверка 2.5a
assert torch.allclose(way1, way2, atol=1e-5); check("way1 == way2", True)
assert torch.allclose(way1, way3, atol=1e-5); check("way1 == way3", True)
print("\n2.5a: OK ✅")

### ✏️ Упражнение 2.5b: Batched attention scores

В стандартном attention:
```text
scores = Q @ K^T / sqrt(d_k)
```

Shapes: Q: (B, H, T, D_head), K: (B, H, T, D_head)
Result: (B, H, T, T)

Реализуйте без einsum.

In [ ]:
# ✏️ YOUR CODE
B, H, T, D_head = 2, 16, 8, 64

Q = torch.randn(B, H, T, D_head)
K = torch.randn(B, H, T, D_head)

# Вычислите attention scores с scaling
scale = ...
scores = ...  # (B, H, T, T)

In [ ]:
# ✅ Проверка 2.5b
shape_eq(scores, (B, H, T, T)); check("scores shape", True)

# Manual check: scores[0, 0, i, j] should equal dot(Q[0,0,i], K[0,0,j]) / sqrt(D_head)
manual_score = (Q[0, 0, 0] * K[0, 0, 3]).sum() / math.sqrt(D_head)
assert abs(scores[0, 0, 0, 3].item() - manual_score.item()) < 1e-5; check("manual dot check", True)
print("\n2.5b: OK ✅")

💡 **Совет**: Для batched matmul в 4D тензорах PyTorch автоматически
обрабатывает первые оси как batch. `(B, H, T, D) @ (B, H, D, T)` —
PyTorch видит batch dims `(B, H)` и делает matmul по последним двум.
Transpose последних двух осей: `.transpose(-2, -1)` или для 4D можно
`.permute(0, 1, 3, 2)`.

## 2.6 Einsum — универсальный тензорный контрактор

📖 **Теория**

`torch.einsum` — нотация Эйнштейна для тензорных операций:
```python
torch.einsum("subscripts", tensor1, tensor2, ...)
```

Правила:
- Каждая буква = ось
- Повторяющиеся буквы = суммирование по этой оси (contraction)
- Буквы в выходе = сохранённые оси

Примеры:
```python
"ij,jk->ik"     # матричное умножение
"bshd,bthd->bsht"  # batched dot product по head dim
"bsht,bthd->bshd"  # attention * values
```

**Все 5 einsum-ов из DeepSeek MLA:**
1. `"bshd,bthd->bsht"` — Q·K^T scores (naive mode)
2. `"bsht,bthd->bshd"` — scores·V (naive mode)
3. `"bshd,hdc->bshc"` — Q_nope через absorbed W_kv (absorb mode)
4. `"bshc,btc->bsht"` — absorbed scores
5. `"bshr,btr->bsht"` — rope scores

In [ ]:
# 👀 Пример: einsum vs matmul
B, T, H, D = 2, 8, 4, 16
Q = torch.randn(B, T, H, D)
K = torch.randn(B, T, H, D)

# Attention scores через einsum
scores_ein = torch.einsum("bshd,bthd->bsht", Q, K)
print(f"einsum scores shape: {scores_ein.shape}")

# То же через permute + matmul
Q_perm = Q.permute(0, 2, 1, 3)  # (B, H, T, D)
K_perm = K.permute(0, 2, 1, 3)  # (B, H, T, D)
scores_mm = Q_perm @ K_perm.transpose(-2, -1)  # (B, H, T, T)

# Einsum выдаёт (B, S, H, T), matmul — (B, H, S, T). Сравним через permute:
scores_ein_perm = scores_ein.permute(0, 2, 1, 3)
assert torch.allclose(scores_ein_perm, scores_mm, atol=1e-5)
print("Equivalent ✅")

### ✏️ Упражнение 2.6a: Перепишите einsum как matmul

Дан einsum `"bshd,hdc->bshc"` (absorbed Q_nope проекция).
Перепишите его через `@` / `torch.matmul`.

In [ ]:
# ✏️ YOUR CODE
B, T, H, D_nope, C = 2, 8, 4, 128, 512

q_nope = torch.randn(B, T, H, D_nope)  # (B, T, H, D_nope)
wkv_b_nope = torch.randn(H, D_nope, C) # (H, D_nope, C)

# Einsum reference
ref = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b_nope)

# YOUR CODE: без einsum
# Подсказка: нужен permute чтобы выровнять H, потом batched matmul
result = ...

In [ ]:
# ✅ Проверка 2.6a
shape_eq(result, (B, T, H, C)); check("shape", True)
assert torch.allclose(result, ref, atol=1e-4); check("matches einsum", True)
print("\n2.6a: OK ✅")

### ✏️ Упражнение 2.6b: Напишите einsum по описанию shapes

Дано:
- `kv_cache`: (B, T, C) — сжатый KV
- `q_compressed`: (B, S, H, C) — сжатый Q

Нужно: scores (B, S, H, T) — через contraction по C.

Напишите строку einsum.

In [ ]:
# ✏️ YOUR CODE
B, S, T, H, C = 2, 4, 32, 16, 512

kv_cache = torch.randn(B, T, C)
q_compressed = torch.randn(B, S, H, C)

# Напишите einsum
scores = ...  # shape (B, S, H, T)

In [ ]:
# ✅ Проверка 2.6b
shape_eq(scores, (B, S, H, T)); check("shape", True)

# Manual check: scores[0, 0, 0, 0] = dot(q_compressed[0,0,0,:], kv_cache[0,0,:])
manual = (q_compressed[0, 0, 0] * kv_cache[0, 0]).sum()
assert abs(scores[0, 0, 0, 0].item() - manual.item()) < 1e-4; check("manual dot check", True)
print("\n2.6b: OK ✅")

💡 **Совет**: Einsum медленнее оптимизированного matmul для стандартных паттернов,
но **намного** читабельнее для нестандартных. DeepSeek использует einsum именно
в MLA, где shapes нетривиальны. Для обычного attention (`Q @ K.T`, `attn @ V`)
лучше использовать `@` или `F.scaled_dot_product_attention`.

## 2.7 Outer Product и комплексные числа для RoPE

📖 **Теория**

Rotary Positional Embeddings (RoPE) используют комплексные числа.

```python
# Частоты: геометрическая прогрессия
freqs = 1.0 / (base ** (torch.arange(0, dim, 2) / dim))

# Outer product: позиции × частоты → матрица углов
t = torch.arange(seqlen)
angles = torch.outer(t, freqs)     # (seqlen, dim//2)

# Комплексные экспоненты: e^(i·angle) = cos(angle) + i·sin(angle)
freqs_cis = torch.polar(torch.ones_like(angles), angles)
```

Ключевые функции:
- `torch.outer(a, b)` — внешнее произведение: `(N,)` × `(M,)` → `(N, M)`
- `torch.polar(abs, angle)` — `abs * e^(i·angle)` → complex tensor
- `torch.view_as_complex(x)` — `(..., 2)` real → `(...)` complex
- `torch.view_as_real(x)` — `(...)` complex → `(..., 2)` real

In [ ]:
# 👀 Пример: построение RoPE frequencies
dim = 8
base = 10000.0
seqlen = 16

freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
print(f"Frequencies ({len(freqs)} values): {freqs.tolist()}")

t = torch.arange(seqlen)
angles = torch.outer(t, freqs)
print(f"Angles shape: {angles.shape}")

freqs_cis = torch.polar(torch.ones_like(angles), angles)
print(f"Complex freqs shape: {freqs_cis.shape}, dtype: {freqs_cis.dtype}")

# Verify: polar(1, angle) == cos(angle) + i*sin(angle)
manual_real = torch.cos(angles[0])
manual_imag = torch.sin(angles[0])
assert torch.allclose(freqs_cis[0].real, manual_real, atol=1e-6)
assert torch.allclose(freqs_cis[0].imag, manual_imag, atol=1e-6)
print("polar == cos + i*sin ✅")

### ✏️ Упражнение 2.7: Постройте RoPE и примените

1. Постройте `freqs_cis` из параметров.
2. Примените rotary embedding к вектору x: превратите x в комплексные,
   умножьте на freqs_cis, верните в real.

In [ ]:
# ✏️ YOUR CODE
dim = 16      # head dimension (must be even for pairs)
seqlen = 32
base = 10000.0

# 1. Построить freqs_cis (seqlen, dim//2) complex
freqs = ...          # (dim//2,) — frequency vector
t = ...              # (seqlen,) — position indices
angles = ...         # (seqlen, dim//2) — outer product
freqs_cis = ...      # (seqlen, dim//2) — complex exponentials

# 2. Применить rotary embedding
x = torch.randn(seqlen, dim)  # (T, D)

# Step a: view x as complex: (T, D) → (T, D//2, 2) → view_as_complex → (T, D//2)
x_complex = ...

# Step b: multiply by freqs_cis
x_rotated_complex = ...

# Step c: back to real: view_as_real → (T, D//2, 2) → flatten → (T, D)
x_rotated = ...

In [ ]:
# ✅ Проверка 2.7
shape_eq(freqs_cis, (seqlen, dim // 2)); check("freqs_cis shape", True)
assert freqs_cis.is_complex(); check("freqs_cis is complex", True)
shape_eq(x_rotated, (seqlen, dim)); check("x_rotated shape", True)
assert not x_rotated.is_complex(); check("x_rotated is real", True)

# Verify norms preserved (rotation doesn't change magnitude)
orig_norms = x.view(seqlen, -1, 2).norm(dim=-1)
rot_norms = x_rotated.view(seqlen, -1, 2).norm(dim=-1)
assert torch.allclose(orig_norms, rot_norms, atol=1e-5); check("norms preserved", True)
print("\n2.7: OK ✅")

💡 **Совет**: RoPE работает на парах измерений. Каждая пара `(2i, 2i+1)`
превращается в комплексное число, умножается на поворот `e^(i·pos·ω)`,
и возвращается в real. Умножение комплексных чисел = поворот в 2D,
поэтому нормы сохраняются. В DeepSeek `apply_rotary_emb` делает
`.float()` перед view_as_complex — это обязательно, т.к. complex64
требует float32 входы.

---
# Глава 3 — Advanced Indexing & Masking

> **Цель**: уверенно использовать slice, boolean mask, gather, scatter,
> topk, where, bincount — всё, что делает MoE routing и attention masking рабочими.

## 3.1 Базовый slicing

📖 **Теория**

```python
x[start:end]          # элементы [start, end)
x[:, start:end]       # по второй оси
x[..., -1]            # последний элемент по последней оси
x[:n]                 # первые n
x[-n:]                # последние n
```

Паттерн KV-кэша из DeepSeek:
```python
self.k_cache[:bsz, start_pos:end_pos] = k
# Записываем k в кэш: первые bsz батчей, позиции [start_pos, end_pos)
```

### ✏️ Упражнение 3.1: KV-cache slicing

Реализуйте запись и чтение из KV-кэша.

In [ ]:
# ✏️ YOUR CODE
max_bsz, max_seq, D = 8, 128, 64
cache = torch.zeros(max_bsz, max_seq, D)

# Simulated first forward pass: bsz=2, positions 0..7
bsz = 2
start_pos = 0
new_keys = torch.randn(bsz, 8, D)  # 8 new tokens

# 1. Запишите new_keys в кэш
# YOUR CODE HERE

# Simulated second forward pass: bsz=2, positions 8..8 (1 new token)
start_pos2 = 8
new_key_single = torch.randn(bsz, 1, D)

# 2. Запишите new_key_single в кэш
# YOUR CODE HERE

# 3. Прочитайте ВСЕ ключи для текущих батчей (позиции 0..8 включительно)
end_pos = 9
all_keys = ...  # (2, 9, 64)

In [ ]:
# ✅ Проверка 3.1
# Check first write
assert torch.equal(cache[:bsz, 0:8], new_keys); check("first write", True)

# Check second write
assert torch.equal(cache[:bsz, 8:9], new_key_single); check("second write", True)

# Check read
shape_eq(all_keys, (2, 9, D)); check("read shape", True)
assert torch.equal(all_keys[:, :8], new_keys); check("read first 8", True)
assert torch.equal(all_keys[:, 8:9], new_key_single); check("read token 9", True)

# Unused batches should still be zero
assert torch.all(cache[2:] == 0); check("unused batches zero", True)
print("\n3.1: OK ✅")

💡 **Совет**: KV-кэш в DeepSeek выделяется на максимальные batch_size и seq_len,
а используется через slicing. Это избегает динамической аллокации памяти во время
генерации и критично для производительности inference.

## 3.2 Boolean Masking

📖 **Теория**

```python
mask = x > 0           # boolean tensor
x[mask]                # выбрать элементы где True → 1D
x[mask] = 0            # записать 0 по маске
```

Паттерн из DeepSeek `ParallelEmbedding`:
```python
mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)
x = x - self.vocab_start_idx
x[mask] = 0          # out-of-range tokens → 0
y = F.embedding(x, self.weight)
y[mask] = 0          # zero out embeddings for out-of-range tokens
```

### ✏️ Упражнение 3.2: Parallel Embedding masking

Реализуйте логику ParallelEmbedding для rank=1, world_size=4, vocab_size=100.

In [ ]:
# ✏️ YOUR CODE
vocab_size = 100
world_size = 4
current_rank = 1

part_vocab_size = vocab_size // world_size  # 25
vocab_start = current_rank * part_vocab_size  # 25
vocab_end = vocab_start + part_vocab_size     # 50

# Пример входных token IDs (некоторые в нашем диапазоне, некоторые нет)
tokens = torch.tensor([10, 30, 45, 60, 25, 49, 50, 99])

# 1. Создайте boolean mask: True для токенов ВНЕ нашего диапазона [25, 50)
out_of_range = ...

# 2. Сдвиньте токены на vocab_start
tokens_shifted = ...

# 3. Обнулите out-of-range токены
# YOUR CODE (in-place на tokens_shifted)

# 4. Fake embedding lookup (просто индексируем случайную матрицу)
weight = torch.randn(part_vocab_size, 16)
embeddings = F.embedding(tokens_shifted, weight)

# 5. Обнулите embeddings для out-of-range токенов
# YOUR CODE (in-place на embeddings)

In [ ]:
# ✅ Проверка 3.2
assert out_of_range.tolist() == [True, False, False, True, False, False, True, True]; check("mask", True)
assert tokens_shifted[0] == 0; check("OOR token zeroed", True)
assert tokens_shifted[1] == 5; check("shifted correctly (30-25=5)", True)
assert torch.all(embeddings[0] == 0); check("OOR embedding zeroed", True)
assert not torch.all(embeddings[1] == 0); check("valid embedding preserved", True)
print("\n3.2: OK ✅")

## 3.3 `masked_fill_` и Attention Masks

📖 **Теория**

```python
x.masked_fill_(bool_mask, value)  # in-place: где mask==True → value
```

Два паттерна из DeepSeek:

1. **Causal mask** (Transformer.forward):
```python
mask = torch.full((seqlen, seqlen), float("-inf")).triu_(1)
```
`triu_(1)` — верхний треугольник без диагонали → `-inf`

2. **Group masking** (Gate.forward):
```python
scores = scores.masked_fill_(mask.unsqueeze(-1), float("-inf"))
```
Зануляет scores невыбранных групп экспертов.

### ✏️ Упражнение 3.3: Causal mask + softmax

Постройте causal mask, примените к scores, вычислите attention weights.

In [ ]:
# ✏️ YOUR CODE
T = 6

# 1. Создайте causal mask (T, T): 0 на/ниже диагонали, -inf выше
causal = ...

# 2. Примените к random scores
scores = torch.randn(T, T)
masked = scores + causal

# 3. Softmax по dim=-1
attn_weights = F.softmax(masked, dim=-1)

In [ ]:
# ✅ Проверка 3.3
shape_eq(causal, (T, T)); check("causal shape", True)

# Позиция 0 видит только себя
assert attn_weights[0, 0] > 0; check("pos 0 attends to self", True)
assert abs(attn_weights[0, 1:].sum().item()) < 1e-6; check("pos 0 doesn't attend to future", True)

# Последняя позиция видит всех
assert (attn_weights[-1] > 0).all(); check("last pos attends to all", True)

# Каждая строка суммируется в 1
assert torch.allclose(attn_weights.sum(dim=-1), torch.ones(T)); check("rows sum to 1", True)
print("\n3.3: OK ✅")

## 3.4 `gather` и `scatter_`

📖 **Теория**

`gather` — выбрать элементы по индексам вдоль оси:
```python
# out[i][j] = input[i][index[i][j]]  (dim=1)
selected = scores.gather(dim=1, index=indices)
```

`scatter_` — обратная операция, разместить значения по индексам:
```python
# self[i][index[i][j]] = src  (dim=1)
mask.scatter_(dim=1, index=indices, value=False)
```

DeepSeek Gate:
```python
# 1. TopK → индексы лучших групп
indices = group_scores.topk(self.topk_groups, dim=-1)[1]

# 2. Scatter: создать mask, пометить выбранные группы как False
mask = scores.new_ones(x.size(0), self.n_groups, dtype=bool)
mask.scatter_(1, indices, False)  # False = "не маскировать"

# 3. Gather: выбрать оригинальные scores по topk indices
weights = original_scores.gather(1, indices)
```

### ✏️ Упражнение 3.4: Gate gather/scatter pattern

Реализуйте полный паттерн: topk → scatter mask → gather weights.

In [ ]:
# ✏️ YOUR CODE
N = 4           # tokens
n_experts = 8   # total experts
topk = 3        # activate top-3

# Scores для каждого эксперта
scores = torch.randn(N, n_experts)

# 1. Найдите top-3 эксперта для каждого токена
topk_values, topk_indices = ...  # shapes: (N, 3), (N, 3)

# 2. Создайте boolean mask (N, n_experts), True везде
# Затем scatter_ False в позиции выбранных экспертов
expert_mask = ...
# YOUR CODE: scatter False at topk_indices

# 3. Gather оригинальные scores для выбранных экспертов
selected_scores = ...  # (N, 3)

In [ ]:
# ✅ Проверка 3.4
shape_eq(topk_indices, (N, topk)); check("topk_indices shape", True)
shape_eq(expert_mask, (N, n_experts)); check("mask shape", True)
shape_eq(selected_scores, (N, topk)); check("selected_scores shape", True)

# Ровно topk=3 False в каждой строке маски
false_counts = (~expert_mask).sum(dim=1)
assert torch.all(false_counts == topk); check("3 experts selected per token", True)

# Gathered scores должны совпадать с topk values
assert torch.allclose(selected_scores, topk_values); check("gathered == topk values", True)
print("\n3.4: OK ✅")

💡 **Совет**: `gather` и `scatter_` — обратные операции. В DeepSeek Gate
они используются вместе: `scatter_` строит маску «какие группы экспертов
оставить», а `gather` выбирает оригинальные (не masked) scores для нормализации.
Это элегантнее, чем boolean indexing, потому что сохраняет фиксированную форму тензора.

## 3.5 `topk`, `where`, `bincount` — MoE dispatch

📖 **Теория**

```python
torch.topk(x, k, dim)    # → (values, indices) — top-k элементов
torch.where(condition)    # → tuple of index tensors (как np.where)
torch.bincount(x)         # → count occurrences of each value
```

MoE dispatch loop в DeepSeek:
```python
weights, indices = self.gate(x)           # (N, topk) routing
counts = torch.bincount(indices.flatten(),
                        minlength=n_experts)
for i in range(start, end):
    if counts[i] == 0: continue           # skip unused experts
    idx, top = torch.where(indices == i)  # find tokens routed to expert i
    y[idx] += expert(x[idx]) * weights[idx, top, None]
```

### ✏️ Упражнение 3.5: Mini MoE dispatch

Реализуйте диспетчеризацию: для каждого эксперта найдите назначенные токены,
примените «эксперт» (просто умножение на 2) и взвесьте результат.

In [ ]:
# ✏️ YOUR CODE
N, D = 8, 16        # 8 tokens, dim=16
n_experts = 4
topk = 2

x = torch.randn(N, D)

# Simulated gate output
torch.manual_seed(123)
weights = torch.rand(N, topk)                          # (8, 2)
indices = torch.randint(0, n_experts, (N, topk))        # (8, 2)

# Initialize output
y = torch.zeros_like(x)  # (N, D)

# 1. Count how many tokens go to each expert
counts = ...  # (n_experts,)

# 2. Dispatch loop
for expert_id in range(n_experts):
    if counts[expert_id] == 0:
        continue
    # a. Find which (token_idx, slot_idx) pairs route to this expert
    token_idx, slot_idx = ...  # torch.where(...)

    # b. Get the tokens and their routing weights
    expert_input = x[token_idx]           # (num_routed, D)
    expert_weights = weights[token_idx, slot_idx]  # (num_routed,)

    # c. "Expert" computation (just multiply by 2 for simplicity)
    expert_output = expert_input * 2      # (num_routed, D)

    # d. Weighted accumulation into y
    # YOUR CODE: y[token_idx] += expert_output * weights[..., None]

In [ ]:
# ✅ Проверка 3.5
shape_eq(y, (N, D)); check("y shape", True)

# Verify: each token should have contributions from exactly topk experts
# Manual check for token 0
token0_experts = indices[0]  # which experts token 0 goes to
token0_weights = weights[0]  # with what weights
expected_y0 = sum(
    (x[0] * 2) * token0_weights[k]
    for k in range(topk)
)
assert torch.allclose(y[0], expected_y0, atol=1e-5); check("token 0 correct", True)
print("\n3.5: OK ✅")

## 3.6 `torch.split` и `torch.cat`

📖 **Теория**

```python
torch.split(tensor, split_sizes, dim)  # разделить по указанным размерам
torch.cat(tensors, dim)                 # склеить обратно
```

DeepSeek MLA — основной паттерн:
```python
# Разделить Q на nope и rope части:
q_nope, q_pe = torch.split(q, [qk_nope_head_dim, qk_rope_head_dim], dim=-1)

# Разделить KV projection:
kv, k_pe = torch.split(kv, [kv_lora_rank, qk_rope_head_dim], dim=-1)

# Собрать обратно:
q = torch.cat([q_nope, q_pe], dim=-1)
k = torch.cat([k_nope, k_pe_expanded], dim=-1)
```

### ✏️ Упражнение 3.6: MLA split/cat round-trip

Реализуйте разделение и сборку Q вектора, как в DeepSeek MLA.

In [ ]:
# ✏️ YOUR CODE
B, T, H = 2, 8, 16
qk_nope_dim = 128
qk_rope_dim = 64
qk_head_dim = qk_nope_dim + qk_rope_dim  # 192

# Полный Q после view
q = torch.randn(B, T, H, qk_head_dim)

# 1. Split на nope и rope
q_nope, q_pe = ...  # split по dim=-1

# 2. "Обработка" (просто умножим rope на 2 для демо)
q_pe_processed = q_pe * 2

# 3. Cat обратно
q_reconstructed = ...  # cat по dim=-1

In [ ]:
# ✅ Проверка 3.6
shape_eq(q_nope, (B, T, H, qk_nope_dim)); check("q_nope shape", True)
shape_eq(q_pe, (B, T, H, qk_rope_dim)); check("q_pe shape", True)
shape_eq(q_reconstructed, (B, T, H, qk_head_dim)); check("reconstructed shape", True)

# nope part should be unchanged
assert torch.equal(q_reconstructed[..., :qk_nope_dim], q_nope); check("nope unchanged", True)
# rope part should be doubled
assert torch.allclose(q_reconstructed[..., qk_nope_dim:], q_pe * 2); check("rope processed", True)
print("\n3.6: OK ✅")

💡 **Совет**: `torch.split` возвращает tuple тензоров, которые **разделяют память**
с оригиналом (как view). Это важно для производительности — нет копирования.
В DeepSeek split используется минимум 3 раза в MLA.forward:
разделение Q, разделение KV projection, разделение KV на k_nope и v.

---
# Глава 4 — nn.Module Structures

> **Цель**: строить модули, которые зеркалят паттерны DeepSeek —
> Parameter, Buffer, Custom Linear, RMSNorm, Residual Connections.

## 4.1 `nn.Parameter` и `register_buffer`

📖 **Теория**

| Что | Назначение | В `.parameters()` | В `.state_dict()` |
|-----|-----------|-------------------|-------------------|
| `nn.Parameter(tensor)` | Обучаемые веса | ✅ | ✅ |
| `register_buffer("name", tensor)` | Необучаемое состояние | ❌ | ✅ (default) |
| `register_buffer("name", tensor, persistent=False)` | Временное | ❌ | ❌ |

DeepSeek:
```python
# Обучаемые:
self.weight = nn.Parameter(torch.empty(out_features, in_features))

# Неперсистентные (кэши, precomputed):
self.register_buffer("k_cache", torch.zeros(...), persistent=False)
self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)
```

### ✏️ Упражнение 4.1: Модуль с Parameter и Buffer

Создайте модуль `ScaleAndShift`:
- `scale` — обучаемый parameter (вектор длины dim)
- `shift` — обучаемый parameter (вектор длины dim)
- `running_mean` — buffer (persistent=True) для EMA среднего
- `step_count` — buffer (persistent=False) для подсчёта шагов

In [ ]:
# ✏️ YOUR CODE
class ScaleAndShift(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        raise NotImplementedError

    def forward(self, x: Tensor) -> Tensor:
        """Apply scale and shift: output = x * scale + shift"""
        raise NotImplementedError

In [ ]:
# ✅ Проверка 4.1
m = ScaleAndShift(64)

param_names = [n for n, _ in m.named_parameters()]
buffer_names = [n for n, _ in m.named_buffers()]
state_keys = list(m.state_dict().keys())

check("scale is parameter", "scale" in param_names)
check("shift is parameter", "shift" in param_names)
check("running_mean is buffer", "running_mean" in buffer_names)
check("step_count is buffer", "step_count" in buffer_names)
check("2 parameters", len(param_names) == 2)
check("running_mean in state_dict", "running_mean" in state_keys)
check("step_count NOT in state_dict", "step_count" not in state_keys)

x = torch.randn(2, 4, 64)
y = m(x)
shape_eq(y, (2, 4, 64)); check("forward shape", True)
print("\n4.1: OK ✅")

## 4.2 Custom Linear и `F.linear`

📖 **Теория**

`nn.Linear` хранит вес `(out_features, in_features)` и опциональный bias `(out_features,)`.
Forward: `F.linear(x, self.weight, self.bias)`.

В DeepSeek используется **кастомный** Linear:
```python
class Linear(nn.Module):
    def __init__(self, in_features, out_features, bias=False, dtype=None):
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter("bias", None)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)
```

`register_parameter("bias", None)` — говорит PyTorch: «bias существует как атрибут,
но его значение None (нет bias)». Это нужно для корректной сериализации.

### ✏️ Упражнение 4.2: Свой Linear

Реализуйте `MyLinear`, который работает идентично `nn.Linear`.

In [ ]:
# ✏️ YOUR CODE
class MyLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        raise NotImplementedError

    def forward(self, x: Tensor) -> Tensor:
        raise NotImplementedError

In [ ]:
# ✅ Проверка 4.2
mine = MyLinear(64, 128, bias=True)
ref = nn.Linear(64, 128, bias=True)

# Copy weights for comparison
with torch.no_grad():
    ref.weight.copy_(mine.weight)
    ref.bias.copy_(mine.bias)

x = torch.randn(2, 4, 64)
y_mine = mine(x)
y_ref = ref(x)
shape_eq(y_mine, (2, 4, 128)); check("shape", True)
assert torch.allclose(y_mine, y_ref, atol=1e-5); check("matches nn.Linear", True)

# Test without bias
mine_nb = MyLinear(32, 16, bias=False)
check("no bias → bias is None", mine_nb.bias is None)
print("\n4.2: OK ✅")

## 4.3 RMSNorm

📖 **Теория**

RMSNorm — упрощённый LayerNorm без центрирования (вычитания mean):

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma$$

где $\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}$

DeepSeek использует `F.rms_norm`:
```python
def forward(self, x):
    return F.rms_norm(x, (self.dim,), self.weight, self.eps)
```

### ✏️ Упражнение 4.3: RMSNorm вручную

Реализуйте RMSNorm без использования `F.rms_norm`.

In [ ]:
# ✏️ YOUR CODE
class MyRMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        raise NotImplementedError

    def forward(self, x: Tensor) -> Tensor:
        """
        x: (..., dim)
        return: (..., dim) — normalized and scaled
        """
        raise NotImplementedError

In [ ]:
# ✅ Проверка 4.3
dim = 64
mine = MyRMSNorm(dim)

# DeepSeek reference
ref = nn.Module()
ref.dim = dim
ref.weight = mine.weight  # share weights
ref.eps = 1e-6
ref.forward = lambda x: F.rms_norm(x, (dim,), mine.weight, 1e-6)

x = torch.randn(2, 8, dim)
y_mine = mine(x)
y_ref = ref.forward(x)

shape_eq(y_mine, (2, 8, dim)); check("shape", True)
assert torch.allclose(y_mine, y_ref, atol=1e-5); check("matches F.rms_norm", True)

# RMSNorm output should have ~unit RMS per vector
rms_out = y_mine.pow(2).mean(dim=-1).sqrt()
# (не точно 1.0 из-за learnable weight, но близко для weight=ones)
check("RMS close to 1", (rms_out - 1.0).abs().mean().item() < 0.1)
print("\n4.3: OK ✅")

💡 **Совет**: RMSNorm вычислительно дешевле LayerNorm (нет вычитания mean)
и стала стандартом в LLM (LLaMA, DeepSeek, Gemma). В DeepSeek RMSNorm
применяется **перед** каждым sublayer (pre-norm): `self.attn(self.attn_norm(x))`.

## 4.4 Residual Connections (Pre-Norm)

📖 **Теория**

Pre-norm residual pattern из DeepSeek Block:
```python
x = x + self.attn(self.attn_norm(x), start_pos, freqs_cis, mask)
x = x + self.ffn(self.ffn_norm(x))
```

Формула:
```text
x = x + Sublayer(Norm(x))
```

Residual connection позволяет градиенту течь напрямую через слои,
решая проблему vanishing gradients в глубоких сетях.

### ✏️ Упражнение 4.4: Mini Transformer Block

Постройте упрощённый Block с pre-norm residual connections.

In [ ]:
# ✏️ YOUR CODE
class MiniBlock(nn.Module):
    """Simplified transformer block with pre-norm residuals."""
    def __init__(self, dim: int, inter_dim: int):
        super().__init__()
        raise NotImplementedError
        # Нужны: attn_norm (RMSNorm), ffn_norm (RMSNorm),
        #        attn (nn.Linear как заглушка), ffn (nn.Linear)

    def forward(self, x: Tensor) -> Tensor:
        """
        x: (B, T, D) -> (B, T, D)
        Apply: x = x + attn(attn_norm(x))
               x = x + ffn(ffn_norm(x))
        """
        raise NotImplementedError

In [ ]:
# ✅ Проверка 4.4
dim = 64
block = MiniBlock(dim, dim * 4)

x = torch.randn(2, 8, dim)
y = block(x)
shape_eq(y, (2, 8, dim)); check("output shape", True)

# Residual: output should not be zero even if sublayers output zeros
# (because of the residual connection x + sublayer(x))
assert y.abs().sum() > 0; check("non-zero output", True)

# Verify gradient flow through residual
x_grad = torch.randn(2, 8, dim, requires_grad=True)
y_grad = block(x_grad)
y_grad.sum().backward()
assert x_grad.grad is not None; check("gradient flows", True)
assert x_grad.grad.abs().sum() > 0; check("non-zero gradient", True)
print("\n4.4: OK ✅")

## 4.5 In-place операции

📖 **Теория**

Суффикс `_` означает in-place модификацию:

| In-place | Out-of-place |
|----------|-------------|
| `x.add_(y)` | `x + y` |
| `x.mul_(s)` | `x * s` |
| `x.clamp_(min, max)` | `x.clamp(min, max)` |
| `x.triu_(diag)` | `x.triu(diag)` |
| `x.masked_fill_(m, v)` | `x.masked_fill(m, v)` |
| `x.zero_()` | `torch.zeros_like(x)` |

⚠️ **Важно**: in-place операции НЕ создают новый тензор.
Это экономит память, но ломает autograd если тензор нужен для backward.

DeepSeek примеры:
```python
mask.triu_(1)              # in-place: верхний треугольник
scores.masked_fill_(...)   # in-place: заполнение по маске
g.data.mul_(clip_coef)     # in-place: через .data обходит autograd
y += bias                  # in-place add
```

### ✏️ Упражнение 4.5: In-place vs out-of-place

Покажите разницу и подводные камни.

In [ ]:
# ✏️ YOUR CODE

# 1. Создайте матрицу и примените triu_ in-place
m = torch.ones(4, 4)
# YOUR CODE: обнулите нижний треугольник (под диагональю) in-place
# Подсказка: triu_(0) оставляет диагональ и выше

# 2. In-place clamp: ограничьте значения в [-0.5, 0.5]
v = torch.randn(10)
v_original = v.clone()  # save copy
# YOUR CODE: clamp v in-place

# 3. Покажите, что in-place меняет ТОТ ЖЕ объект
a = torch.tensor([1.0, 2.0, 3.0])
b = a                    # b is the SAME tensor (alias)
a.mul_(2)
same_object = ...        # True если b тоже изменился

In [ ]:
# ✅ Проверка 4.5
# triu check
assert m[1, 0] == 0 and m[0, 0] == 1; check("triu_ diagonal preserved", True)
assert m[2, 0] == 0 and m[2, 1] == 0; check("triu_ below zeroed", True)
assert m[0, 3] == 1; check("triu_ above preserved", True)

# clamp check
assert v.min() >= -0.5 and v.max() <= 0.5; check("clamp_ bounds", True)

# aliasing check
assert same_object == True; check("in-place affects aliases", True)
assert torch.equal(a, b); check("a and b are same tensor", True)
print("\n4.5: OK ✅")

💡 **Совет**: In-place операции через `.data` (как `g.data.mul_(clip_coef)`)
обходят autograd tracking. В DeepSeek gradient clipping делается именно так —
после backward, перед optimizer step. Никогда не делайте in-place на тензорах
с `requires_grad=True` до backward — это сломает граф вычислений.

---
# 🎉 Лабораторная завершена!

Вы изучили все ключевые PyTorch-паттерны, необходимые для чтения и написания
кода уровня DeepSeek V3:

| Глава | Что освоили | Паттерны DeepSeek |
|-------|------------|-------------------|
| **1. Foundations** | Создание, shapes, view, squeeze, dtype, device | KV cache, head split, MLA view, freqs_cis broadcast |
| **2. Operations** | Element-wise, broadcast, reductions, matmul, einsum, RoPE | SwiGLU, expand, Gate norm, attention scores, all 5 MLA einsums |
| **3. Indexing** | Slicing, boolean mask, masked_fill, gather/scatter, topk/where | KV cache, ParallelEmbedding, causal mask, Gate routing, MoE dispatch |
| **4. Modules** | Parameter, buffer, Linear, RMSNorm, residuals, in-place | Custom Linear, pre-norm Block, register_buffer |

**Следующий шаг**: откройте `model.py` DeepSeek V3 и прочитайте его от начала до конца.
Каждый паттерн из этой лабораторной встретится вам в реальном коде.